# RAG – Fichas de Datos de Seguridad CORONA
**Google Colab — ejecutar con GPU T4 (Runtime > Change runtime type > T4 GPU)**

Flujo:
1. Instalar dependencias
2. Subir los `.md` generados localmente
3. Indexar en ChromaDB con fastembed (mismo modelo que local)
4. Instalar Ollama + modelo LLM
5. Consultas con trazabilidad
6. Exportar DB para uso local

## Celda 1 — Instalar dependencias

In [1]:
!pip install -q fastembed chromadb rank-bm25 ollama
print('Dependencias instaladas OK')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.6/116.6 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 68.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 99.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 83.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.9/323.9 kB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.

## Celda 2 — Subir archivos .md

Comprime localmente con: `zip -r markdown_corona.zip output/markdown/`
Luego sube el zip aquí.

In [2]:
from google.colab import files
import zipfile, os
from pathlib import Path

print('Sube el archivo markdown_corona.zip')
uploaded = files.upload()

for fname in uploaded:
    with zipfile.ZipFile(fname, 'r') as z:
        z.extractall('.')

md_files = list(Path('output/markdown').glob('*.md'))
print(f'Archivos .md cargados: {len(md_files)}')
for f in sorted(md_files):
    print(f'  {f.name} ({f.stat().st_size // 1024} KB)')

Sube el archivo markdown_corona.zip


Saving markdown_corona.zip to markdown_corona.zip
Archivos .md cargados: 17
  FDS 29 - PINTURA PRIMERA MANO & ACABADO - CORONA .md (69 KB)
  FDS 31 - 407141521 - RECUBRIMIENTO ANTIGRAFFITI - CORONA .md (78 KB)
  FDS 42 - PINTURA LAVABLE ANTIBACTERIAL - CORONA .md (70 KB)
  FDS 43 - PINTURA PRIMERA MANO & ACABADO - CORONA .md (68 KB)
  FDS 44 - TEXTUCO - CORONA .md (69 KB)
  FDS 49 - PINTURA TOTAL - CORONA .md (72 KB)
  FDS 61 - PINTURA SEN╠âALIZACIO╠üN Y DEMARCACIO╠üN - CORONA.md (79 KB)
  FDS 67 - PINTURA EXTERIORES  - CORONA.md (87 KB)
  FDS 68 - PINTURA SUPERLAVABLE ZERO - CORONA.md (86 KB)
  FDS 75 -PINTURA-ALTA-COBERTURA - CORONA.md (79 KB)
  FDS 76 - PINTURA TOTAL - CORONA.md (78 KB)
  FDS 88 - PINTURA LAVABLE - CORONA.md (85 KB)
  FDS 89 - PINTURA LAVABLE BIO - CORONA.md (88 KB)
  FDS 91 - PINTURA EXTERIORES - CORONA.md (83 KB)
  FDS 92 - PINTURA COOLGUARD - CORONA.md (89 KB)
  FDS 93 - PINTURA FACHADA FLEXIBLE - CORONA.md (84 KB)
  FDS 94 - ESMALTE METAL MASTER PREMIUM - CORONA

## Celda 3 — Fragmentar documentos (chunking)

In [3]:
import re, json
from pathlib import Path

FABRICANTE = 'CORONA'
MD_DIR = 'output/markdown'
MAX_WORDS, OVERLAP_WORDS = 800, 100

SECTION_RE = re.compile(r'^#{1,3}\s+SECCI[ÓO]N\s+(\d{1,2})[:.\s]*(.*?)$', re.MULTILINE | re.IGNORECASE)
TABLE_RE   = re.compile(r'(\|[^\n]+\|\n\|[-:| ]+\|\n(?:\|[^\n]+\|\n)+)', re.MULTILINE)

def split_overlap(text, max_w, ov):
    words = text.split()
    out, start = [], 0
    while start < len(words):
        end = min(start + max_w, len(words))
        out.append(' '.join(words[start:end]))
        if end == len(words): break
        start = end - ov
    return out

def chunk_md(md_path, fabricante):
    text = Path(md_path).read_text(encoding='utf-8')
    doc  = Path(md_path).stem
    cnt  = 0
    chunks = []

    def make(sn, st, content, tipo='texto'):
        nonlocal cnt
        content = content.strip()
        if not content: return None
        cnt += 1
        return dict(chunk_id=f'{doc}_sec{sn or 0}_{tipo}_{cnt}',
                    documento=doc, fabricante=fabricante,
                    seccion_num=sn, seccion_titulo=st,
                    tipo=tipo, pagina=None, texto=content)

    hdrs = [(m.start(), int(m.group(1)), m.group(2).strip()) for m in SECTION_RE.finditer(text)]
    hdrs.append((len(text), None, None))

    if hdrs[0][0] > 0:
        c = make(None, 'Encabezado', text[:hdrs[0][0]])
        if c: chunks.append(c)

    for i in range(len(hdrs)-1):
        pos, sn, st = hdrs[i]
        nxt = hdrs[i+1][0]
        nl  = text.find('\n', pos)
        body = text[(nl+1 if nl!=-1 else pos):nxt]

        segs, last = [], 0
        for m in TABLE_RE.finditer(body):
            before = body[last:m.start()].strip()
            if before: segs.append(('texto', before))
            segs.append(('tabla', m.group(0).strip()))
            last = m.end()
        tail = body[last:].strip()
        if tail: segs.append(('texto', tail))

        for tipo, seg in segs:
            if tipo == 'tabla':
                c = make(sn, st, seg, 'tabla')
                if c: chunks.append(c)
            elif len(seg.split()) <= MAX_WORDS:
                c = make(sn, st, seg)
                if c: chunks.append(c)
            else:
                for sub in split_overlap(seg, MAX_WORDS, OVERLAP_WORDS):
                    c = make(sn, st, sub)
                    if c: chunks.append(c)
    return chunks

all_chunks = []
for f in sorted(Path(MD_DIR).glob('*.md')):
    fc = chunk_md(str(f), FABRICANTE)
    all_chunks.extend(fc)
    print(f'  {f.name}: {len(fc)} chunks')

print(f'\nTotal: {len(all_chunks)} chunks')
Path('data').mkdir(exist_ok=True)
with open(f'data/chunks_{FABRICANTE.lower()}.json', 'w', encoding='utf-8') as f:
    json.dump(all_chunks, f, ensure_ascii=False)
print('chunks_corona.json guardado')

  FDS 29 - PINTURA PRIMERA MANO & ACABADO - CORONA .md: 70 chunks
  FDS 31 - 407141521 - RECUBRIMIENTO ANTIGRAFFITI - CORONA .md: 65 chunks
  FDS 42 - PINTURA LAVABLE ANTIBACTERIAL - CORONA .md: 62 chunks
  FDS 43 - PINTURA PRIMERA MANO & ACABADO - CORONA .md: 61 chunks
  FDS 44 - TEXTUCO - CORONA .md: 60 chunks
  FDS 49 - PINTURA TOTAL - CORONA .md: 66 chunks
  FDS 61 - PINTURA SEN╠âALIZACIO╠üN Y DEMARCACIO╠üN - CORONA.md: 69 chunks
  FDS 67 - PINTURA EXTERIORES  - CORONA.md: 78 chunks
  FDS 68 - PINTURA SUPERLAVABLE ZERO - CORONA.md: 74 chunks
  FDS 75 -PINTURA-ALTA-COBERTURA - CORONA.md: 68 chunks
  FDS 76 - PINTURA TOTAL - CORONA.md: 68 chunks
  FDS 88 - PINTURA LAVABLE - CORONA.md: 79 chunks
  FDS 89 - PINTURA LAVABLE BIO - CORONA.md: 79 chunks
  FDS 91 - PINTURA EXTERIORES - CORONA.md: 75 chunks
  FDS 92 - PINTURA COOLGUARD - CORONA.md: 75 chunks
  FDS 93 - PINTURA FACHADA FLEXIBLE - CORONA.md: 75 chunks
  FDS 94 - ESMALTE METAL MASTER PREMIUM - CORONA.md: 88 chunks

Total: 1212 

## Celda 4 — Embeddings con fastembed + indexar ChromaDB

In [4]:
import numpy as np
from fastembed import TextEmbedding
import chromadb

MODEL_NAME = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
print(f'Cargando modelo: {MODEL_NAME} ...')
emb_model = TextEmbedding(model_name=MODEL_NAME)
print('Modelo cargado')

texts = [c['texto'] for c in all_chunks]
print(f'Generando embeddings para {len(texts)} chunks...')
raw = list(emb_model.embed(texts, batch_size=128))
embs = np.array(raw, dtype=np.float32)
norms = np.linalg.norm(embs, axis=1, keepdims=True)
embs = embs / np.where(norms == 0, 1, norms)
print(f'Embeddings shape: {embs.shape}')

Cargando modelo: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2 ...


/tmp/ipykernel_9864/2111485923.py:7: UserWarning: The model sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2 now uses mean pooling instead of CLS embedding. In order to preserve the previous behaviour, consider either pinning fastembed version to 0.5.1 or using `add_custom_model` functionality.
  emb_model = TextEmbedding(model_name=MODEL_NAME)
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Modelo cargado
Generando embeddings para 1212 chunks...
Embeddings shape: (1212, 384)


In [5]:
Path('data/chroma_db').mkdir(parents=True, exist_ok=True)
client     = chromadb.PersistentClient(path='data/chroma_db')
collection = client.get_or_create_collection(
    name=f'fds_{FABRICANTE.lower()}',
    metadata={'hnsw:space': 'cosine'}
)

ids   = [c['chunk_id'] for c in all_chunks]
metas = [{k: (str(v) if v is not None else '') for k,v in c.items() if k not in ('texto','chunk_id')} for c in all_chunks]
BATCH = 400

for i in range(0, len(all_chunks), BATCH):
    collection.upsert(
        ids=ids[i:i+BATCH],
        embeddings=embs[i:i+BATCH].tolist(),
        documents=texts[i:i+BATCH],
        metadatas=metas[i:i+BATCH]
    )
    print(f'  {min(i+BATCH, len(all_chunks))}/{len(all_chunks)} indexados')

print(f'Colección fds_corona: {collection.count()} chunks indexados')

  400/1212 indexados
  800/1212 indexados
  1200/1212 indexados
  1212/1212 indexados
Colección fds_corona: 1212 chunks indexados


## Celda 5 — Instalar Ollama + modelo LLM

In [6]:
import subprocess, time

# zstd es requerido por el instalador de Ollama en Colab
!apt-get install -qq -y zstd

!curl -fsSL https://ollama.ai/install.sh | sh

proc = subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(4)
print('Servidor Ollama iniciado')


Selecting previously unselected package zstd.
(Reading database ... 122363 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
Servidor Ollama iniciado


In [10]:
# qwen2.5:7b (~4.7 GB) — mejor calidad en español
# qwen2.5:3b (~2 GB)   — alternativa si Colab tiene poco disco
!ollama pull qwen2.5:7b

## Celda 6 — Sistema RAG (retriever + generador)

In [11]:
import re as _re, ollama
from rank_bm25 import BM25Okapi

def tokenize(t): return _re.findall(r'\w+', t.lower())

bm25 = BM25Okapi([tokenize(c['texto']) for c in all_chunks])

def dense_search(query, n=10):
    q = np.array(list(emb_model.embed([query])), dtype=np.float32)[0]
    q = q / np.linalg.norm(q)
    res = collection.query(query_embeddings=[q.tolist()], n_results=min(n, collection.count()),
                           include=['documents','metadatas','distances'])
    return [{'texto': d, 'dense_score': float(1-dist), **m}
            for d, m, dist in zip(res['documents'][0], res['metadatas'][0], res['distances'][0])]

def bm25_search(query, n=10):
    scores = bm25.get_scores(tokenize(query))
    top = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:n]
    return [{'bm25_score': float(scores[i]), **all_chunks[i]} for i in top if scores[i] > 0]

def rrf(rankings, k=60):
    scores, docs = {}, {}
    for rank_list in rankings:
        for rank, doc in enumerate(rank_list):
            cid = doc.get('chunk_id', doc.get('texto','')[:60])
            scores[cid] = scores.get(cid, 0.) + 1./(k+rank+1)
            docs[cid] = doc
    out = []
    for cid in sorted(scores, key=scores.__getitem__, reverse=True):
        e = dict(docs[cid]); e['rrf_score'] = round(scores[cid], 6); out.append(e)
    return out

def search(query, k=5):
    d = dense_search(query, n=k*2)
    b = bm25_search(query, n=k*2)
    return (rrf([d, b]) if b else d)[:k]

PROMPT = """Eres un experto en Fichas de Datos de Seguridad (FDS) de pinturas.
Responde usando ÚNICAMENTE el contexto. Cita cada dato con [Documento §SecciónN].
Si no está en el contexto: "No encontrado en los documentos disponibles."

CONTEXTO:
{context}

PREGUNTA: {query}

RESPUESTA:"""

def rag(query, k=5, model='qwen2.5:7b'):
    chunks = search(query, k)
    context = '\n\n---\n\n'.join(
        f"[{i+1}] {c.get('documento','')} | §{c.get('seccion_num','?')} {c.get('seccion_titulo','')}\n{c['texto']}"
        for i, c in enumerate(chunks)
    )
    resp = ollama.chat(model=model,
                       messages=[{'role':'user', 'content': PROMPT.format(context=context, query=query)}],
                       options={'temperature': 0.1, 'num_predict': 1024})
    return resp['message']['content'], chunks

print('RAG listo')

RAG listo


## Celda 7 — Demo de consultas

In [12]:
def ask(query):
    print(f'PREGUNTA: {query}')
    print('-'*60)
    answer, sources = rag(query)
    print(f'RESPUESTA:\n{answer}')
    print('\nFUENTES:')
    for s in sources:
        print(f"  [{s.get('rrf_score',0):.4f}] {s.get('documento','')[:50]} §{s.get('seccion_num','?')}")
    print('='*60 + '\n')

ask('¿Cuál es el punto de inflamación de la Pintura Exteriores CORONA?')

PREGUNTA: ¿Cuál es el punto de inflamación de la Pintura Exteriores CORONA?
------------------------------------------------------------
RESPUESTA:
No encontrado en los documentos disponibles.

FUENTES:
  [0.0325] FDS 94 - ESMALTE METAL MASTER PREMIUM - CORONA §9
  [0.0164] FDS 67 - PINTURA EXTERIORES  - CORONA §10
  [0.0161] FDS 67 - PINTURA EXTERIORES  - CORONA §14
  [0.0159] FDS 42 - PINTURA LAVABLE ANTIBACTERIAL - CORONA  §11
  [0.0159] FDS 67 - PINTURA EXTERIORES  - CORONA §15



In [13]:
ask('¿Qué equipo de protección personal se recomienda para manipular la Pintura Lavable CORONA?')

PREGUNTA: ¿Qué equipo de protección personal se recomienda para manipular la Pintura Lavable CORONA?
------------------------------------------------------------
RESPUESTA:
Para manipular la Pintura Lavable CORONA, se recomienda el uso de los siguientes equipos de protección personal:

- **Protección respiratoria:** Máscara autofiltrante para gases y vapores (Filtro tipo A) con NOMATIVIDAD APLICABLE: NTC 1584, NTC 1589, NTC 3851 y NTC 1728. Reemplazar cuando se detecte olor o sabor del contaminante en el interior de la máscara.

- **Protección específica de las manos:** Guantes NO desechables de protección química con NOMATIVIDAD APLICABLE: NTC 3398, EN 374 y EN420. El tiempo de paso (Breakthrough Time) indicado por el fabricante debe ser superior al del tiempo de uso del producto. No emplear cremas protectoras después del contacto del producto con la piel.

- **Protección ocular y facial:** Pantalla facial con NOMATIVIDAD APLICABLE: NTC 1825, NTC 1826 y ANSI Z87.1. Limpiar a diario y 

In [14]:
ask('¿Qué hacer en caso de derrame accidental de TEXTUCO CORONA?')

PREGUNTA: ¿Qué hacer en caso de derrame accidental de TEXTUCO CORONA?
------------------------------------------------------------
RESPUESTA:
En caso de derrame accidental de TEXTUCO CORONA, se deben seguir las siguientes medidas:

1. Aislar la fuga siempre que no suponga un riesgo adicional para las personas involucradas.
2. Ante la exposición potencial con el producto derramado, es obligatorio el uso de elementos de protección personal (ver sección 8 de la FDS).
3. Evacuar la zona y mantener a las personas sin protección alejadas.

Además, se recomienda:

4. Absorber el vertido mediante arena o absorbente inerte y trasladarlo a un lugar seguro.
5. No absorber en serrín u otros absorbentes combustibles.
6. Para cualquier consideración relativa a la eliminación consultar la sección 13 de la FDS.

Estas medidas están basadas en las recomendaciones generales para el manejo de productos peligrosos y se deben seguir las disposiciones adicionales del Plan de Emergencia Interior y las Fichas

In [15]:
ask('¿Cuáles son las condiciones de almacenamiento recomendadas para el Esmalte Metal Master CORONA?')

PREGUNTA: ¿Cuáles son las condiciones de almacenamiento recomendadas para el Esmalte Metal Master CORONA?
------------------------------------------------------------
RESPUESTA:
Las condiciones de almacenamiento recomendadas para el Esmalte Metal Master CORONA son las siguientes:

- Temperatura mínima: 5 ºC [Documento §2 Identificación de peligros]
- Temperatura máxima: 30 ºC [Documento §2 Identificación de peligros]
- Tiempo máximo: 12 meses [Documento §7 Manipulación y almacenamiento]

Además, se recomienda evitar fuentes de calor, radiación, electricidad estática y el contacto con alimentos [Documento §7 Manipulación y almacenamiento].

FUENTES:
  [0.0452] FDS 49 - PINTURA TOTAL - CORONA  §7
  [0.0325] FDS 76 - PINTURA TOTAL - CORONA §7
  [0.0164] FDS 94 - ESMALTE METAL MASTER PREMIUM - CORONA §2
  [0.0161] FDS 94 - ESMALTE METAL MASTER PREMIUM - CORONA §14
  [0.0159] FDS 49 - PINTURA TOTAL - CORONA  §7



## Celda 8 — Exportar DB para uso local

Descarga `rag_corona_db.zip` y descomprímelo en la carpeta del proyecto.
Con Ollama instalado localmente, el sistema corre sin Colab.

In [16]:
import shutil
from google.colab import files

shutil.make_archive('rag_corona_db', 'zip', '.', 'data')
print('rag_corona_db.zip listo')
files.download('rag_corona_db.zip')

rag_corona_db.zip listo


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>